## **Libraries & Data Loading**

**Loading Libraries**

In [7]:
import numpy as np
import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
import plotly.express as px
import os
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings("ignore")

Agar code se bhi blank screen aaye, toh **pio.renderers.default = 'notebook_connected'** ki line ko hata kar **pio.renderers.default = 'iframe'** ya **'vscode'** try karein. iframe har dafa kaam kar jata hai kyunki yeh graph ko ek alag isolated block mein render karta hai

In [3]:
import plotly.io as pio

# Telling Plotly to render inside vscode
pio.renderers.default = 'vscode'


**2. Loading Dataset By Kaggle CLI**

In [ ]:
# !kaggle datasets download mrayyanshehzad/synthetic-retail-dataset-10-million-transactions -p /content/dataset --unzip

In [ ]:
# base_path = "/content/dataset/retail_contaminated_dataset"  # ya retail_clean_dataset
# files = os.listdir(base_path)
# print("Files:", files)

# csv_files = [f for f in files if f.endswith(".csv")]

# df = {}
# for file in csv_files:
#     file_path = os.path.join(base_path, file)
#     df_name = file.replace('.csv', '')
#     df[df_name] = pd.read_csv(file_path)
#     print(f"Loaded: {file} (Shape: {df[df_name].shape})")

**3. Loading Dataset By Local Memory**

In [4]:
# import  zipfile

# base_path = r"Datasets\retail_contaminated_dataset.zip"

# with zipfile.ZipFile(base_path, 'r' ) as zip_ref:
#     zip_ref.extractall(r"Datasets\extracted_files")


files = os.listdir(r"Datasets\extracted_files")
print("Files:", files)

csv_files = [f for f in files if f.endswith(".csv")]

df = {}
for file in csv_files:
    file_path = os.path.join(r"Datasets\extracted_files", file)
    df_name = file.replace('.csv', '')

    if df_name == "sales_transactions":
        continue  # Skip loading the sales_transactions.csv file
    
    df[df_name] = pd.read_csv(file_path)
    print(f"Loaded: {file} (Shape: {df[df_name].shape})")

Files: ['customer_master.csv', 'inventory_snapshot.csv', 'processed_sales_transactions.parquet', 'promotions.csv', 'sales_transactions.csv', 'sku_inventory_flags.csv', 'sku_master.csv', 'store_master.csv']
Loaded: customer_master.csv (Shape: (10000, 7))
Loaded: inventory_snapshot.csv (Shape: (26408, 6))
Loaded: promotions.csv (Shape: (100, 8))
Loaded: sku_inventory_flags.csv (Shape: (600, 6))
Loaded: sku_master.csv (Shape: (5000, 7))
Loaded: store_master.csv (Shape: (30, 5))


**Loading `processed_sales_transactions` File**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
df['processed_sales_transactions'] = pd.read_parquet(r"Datasets/extracted_files/processed_sales_transactions.parquet")

# df['processed_sales_transactions'] = pd.read_parquet(r"/content/drive/MyDrive/Zidio/processed_sales_transactions.parquet")

In [6]:
df_sales = df['processed_sales_transactions'].iloc[:3000000]

df_sales.shape

(3000000, 30)

## **Day 04 - Exploratory Data Analysis**

In [ ]:
df_sales.head()

In [ ]:
df_sales.info()

In [ ]:
df_sales.describe().round(3)

In [ ]:
px.box(
    df_sales,
    y="total_cost",
    points=False,
    log_y=True,  # Is se scale compress ho kar clear dikhega
    title="Log-Scaled Total Cost Distribution",
)

In [ ]:
# numeric features
features = [
    "quantity",
    "unit_price",
    "total_value",
    "discount_pct",
    "cost_price",
    "unit_margin",
    "margin_pct",
    "total_cost",
    "total_profit",
    "profit_pct",
    "age",
]

In [ ]:
# 2. Normalizing data into 0 & 1
scaler = MinMaxScaler()
df_scaled = df_sales.copy()
df_scaled[features] = scaler.fit_transform(df_sales[features])


In [ ]:
# 3. Melt/Reshape Data
df_reshaped = df_scaled.melt(value_vars=features, var_name="Feature", value_name="Normalized_Value")


In [ ]:
df_reshaped.head()

In [ ]:
# 4. Single Combined Boxplot
fig = px.box(
    df_reshaped,
    x="Feature",
    y="Normalized_Value",
    points='outliers',  # Outliers hide
    color="Feature",
    title="Normalized Distribution Comparison Across All Features (0 to 1 Scale)",
)

fig.update_layout(showlegend=False, xaxis_tickangle=-45)
fig.show()

### **Sales Performance & Trend Analysis (Time-Series)**

In [ ]:
grouped_data= df['processed_sales_transactions'].groupby(['year', 'month', 'quarter','day'])['total_value'].sum().reset_index()

In [ ]:
grouped_data

**By Reveneue**

In [ ]:
px.bar(grouped_data, x='month', y='total_value', animation_frame='year', title='Sales Analysis by Year', labels={'total_value': 'Total Revenue', 'month': 'Month'},
       color='total_value', color_continuous_scale='emrld')

**By Quarter**

In [ ]:
px.bar(grouped_data, x='month', y='total_value', animation_frame='year', title='Sales Analysis by Year', labels={'total_value': 'Total Revenue', 'month': 'Month'},
       color='quarter', color_continuous_scale='emrld')

**Monthly Revenue & Profit (side by side)**

only revenue can't tell about which month we get most profit. Therefore i'm taking profit and revenue side by side

In [ ]:
monthly_data = df['processed_sales_transactions'].groupby(['year','month'], observed=True).agg( revenue = ('total_value', 'sum'), profit = ('total_profit', 'sum')).reset_index()

In [ ]:
px.line( monthly_data, x='month', y=['revenue', 'profit'], title='Monthly Revenue vs Profit by Year', labels={'value': 'Amount', 'month': 'Month'},
         color='year')


### **Customer & Store Behavior Analysis**

**Top 10 Stores by Revenue and Profit**

In [ ]:
stores_data = df['processed_sales_transactions'].groupby('store_id').agg(total_value = ('total_value', 'sum'), total_profit = ('total_profit', 'sum')).reset_index()

In [ ]:
stores_data.head()

In [ ]:
top_stores = df['store_master'].merge(stores_data, on='store_id').sort_values(by='total_value', ascending=False).head(10)
top_stores

In [ ]:
px.bar(
    top_stores,
    x='store_name',
    y='total_value',
    title='Top 10 Stores by Revenue (colored by Profit)',
    labels={'total_value': 'Total Revenue', 'store_name': 'Store Name'},
    color='total_profit',
    color_continuous_scale='emrld'
)

**Quantity per Order**

In [ ]:
df['processed_sales_transactions'].head()

In [ ]:
px.histogram(df['processed_sales_transactions'], x='quantity', labels={'quantity': 'Quantity'}, title='Basket Size / Quantity per Order', color='quantity')

**Quantity vs. Total Value**

In [ ]:
px.scatter(df['processed_sales_transactions'], x='quantity',y='total_value', labels={'quantity': 'Quantity', 'total_value' : 'Total Value'}, title='Quantity vs Total Value')

### **Product & Inventory Insights**

**Top 10 Selling SKUs — by quantity and by profit**

> Using Mini Dataset `df_sales` to reduce power consumption

In [ ]:
skus_data = df_sales.groupby('sku_id').agg(
    quantity = ('quantity', 'sum'),
    total_value = ('total_value', 'sum'),
    total_profit = ('total_profit', 'sum')
).reset_index()


In [ ]:
top_skus= df['sku_master'].merge(skus_data, on='sku_id')

In [ ]:
top_skus_by_qnty=top_skus.sort_values(by='quantity', ascending=False).head(10)
top_skus_by_qnty

In [ ]:
px.bar(
    top_skus_by_qnty,
    x='quantity',
    y='sku_name',
    title='Top 10 Selling SKUs by Quantity (colored by Profit)',
    labels={'quantity': 'Quantity', 'sku_name': 'SKU Name'},
    color='total_profit',
    color_continuous_scale='emrld'
)

Best sellers by quantity aren't always the most profitable.
Therefore we check second way:

In [ ]:
top_skus_by_pft = top_skus.sort_values(by='total_profit', ascending=False).head(10)

px.bar(
    top_skus_by_pft,
    x='total_profit',
    y='sku_name',
    title='Top 10 SKUs by Profit',
    labels={'total_profit': 'Total Profit', 'sku_name': 'SKU Name'},
    color='quantity',
    color_continuous_scale='emrld'
)

**Channel-wise Sales Distribution**

In [ ]:
px.pie(df_sales, names='channel', values='total_value', title='Channel-wise Sales Distribution', hole=0.4)

**Category / subcategory revenue treemap**

In [ ]:
treemap_data = df_sales.groupby(['channel', 'category', 'subcategory'])['total_value'].sum().reset_index()

In [ ]:
px.treemap(
    treemap_data,
    path=['channel', 'category', 'subcategory'],
    values='total_value',
    title='Channel > Category > Subcategory Revenue',
    color='total_value',
)

### **Discount & Promotion Impact Analysis**

In [ ]:
df_sales.head()

In [ ]:
df_sales.info()

In [ ]:
px.histogram(df_sales, x='promo_applied', title='Sales Transactions: Promo vs No Promo', color='promo_applied')

**Discount Percentage vs. Quantity vs. Profit** 
 
> Bigger discount causes more sales but at what cost w.r.t margin?

In [ ]:
px.scatter(
    df_sales,
    x='discount_pct', y='quantity', color='total_profit',
    title='Discount % vs Quantity (colored by total profit)',
    color_continuous_scale='RdYlGn'
)

### **Interdependency of Sales Features**

In [ ]:
corr = df_sales[['quantity', 'unit_price', 'total_value', 'discount_pct', 'total_profit']].corr().round(2)

In [ ]:
px.imshow(
    corr,
    title="Interdependency of Sales Features",
    text_auto=True,
    color_continuous_scale='emrld'
)

### **Region With Most Sales**

In [ ]:
df_sales.head()

In [ ]:
df_sales['city'].value_counts()

In [ ]:
region = df_sales.groupby('city').agg(total_value=('total_value', 'sum'), total_profit=('total_profit', 'sum')).reset_index().sort_values(by='total_value', ascending=False)

px.bar(region, x='total_value', y='city', color='total_profit', color_continuous_scale='emrld',
       title='Revenue by City (colored by profit)')

### **Inventory-Linked Risk Analysis**

(This was loaded but never used before)

As D4 deliverable is stockout/overstock risk scoring. `inventory_snapshot.csv` provides `stock_on_hand`, `reorder_point`, and `safety_stock` per store/SKU 


I build a first, simple, transparent rule from it now, and validate it against the known ground-truth flags.

In [ ]:
df['inventory_snapshot'].head()

In [ ]:
inventory = df['inventory_snapshot'].merge(df['sku_master'][['sku_id', 'sku_name', 'category']], on='sku_id', how='left')

In [ ]:
inventory.head()

**Rules Creation**

Stock Out Risk:

In [ ]:
inventory['stockout_risk'] = inventory['stock_on_hand'] <= inventory['reorder_point']

In [ ]:
print("Stockout Risk flagged rows w.r.t Store/SKU:", inventory['stockout_risk'].sum())

Over Stock Risk:

In [ ]:
inventory['overstock_risk'] = inventory['stock_on_hand'] > (inventory['reorder_point'] + inventory['safety_stock']) * 3

In [ ]:
print("Overstock Risk flagged rows w.r.t Store/SKU:", inventory['overstock_risk'].sum())

In [ ]:
inventory.head()

In [ ]:
risk_summary = inventory.groupby('category')[['stockout_risk', 'overstock_risk']].sum().reset_index()

In [ ]:
px.bar(
    risk_summary,
    x='category',
    y=['stockout_risk', 'overstock_risk'], # Dono columns ko list ma pass karein
    title='Risk Summary by Category',
    barmode='group'
)

**with respect to Stores:**

> using `.melt()` to convert rows into column data.

In [ ]:
inventory_melted = inventory.melt(
    id_vars=['store_id', 'category'], 
    value_vars=['stockout_risk', 'overstock_risk'],
    var_name='Risk_Type', 
    value_name='Has_Risk'
)
inventory_melted.head()

In [ ]:
risky_items = inventory_melted[inventory_melted['Has_Risk'] == True]

In [ ]:
risk_summary = risky_items.groupby(['store_id', 'category', 'Risk_Type']).size().reset_index(name='Risky_Items_Count')


In [ ]:
px.bar(
    risk_summary,
    x='store_id',
    y='Risky_Items_Count',
    color='category',             # Har category ko alag rang milega
    facet_row='Risk_Type',        # Upper panel Stockout ka, lower panel Overstock ka
    title='Category-wise Risky Items for Each Store',
    labels={'Risky_Items_Count': 'Count of Risky Items', 'store_id': 'Store ID', 'category': 'Category'},
    height=700
)


### **Validation**
 Test your independent rule using the `sku_inventory_flags.csv` file as an answer key to score its accuracy without using it for training.  

In [ ]:
flags = df['sku_inventory_flags']
flags.head(350)

##### **Key Points to Note.**
* **Recall  (Sensitivity):** **"Catch everything."** How many of the total actual problems did you find? (Goal: Zero misses).
* **Precision (Exactness):** **"Be 100% sure."** How many of your guesses were actually correct? (Goal: Zero false alarms).
---

* **`hit_...`:** The correct guesses your rule got right.
* **`known_...`:** The real ground-truth answers in the data.
* **`our_...`:** The total guesses made by your rule.

---
- `set()`: unique values
- `.loc[:,:]`

**Actual Data** - Check in sku_inventory_flags file: 

In [ ]:
known_stockout_skus = set(flags.loc[flags['flag'] == 'STOCKOUT_RISK', 'sku_id'])
known_slowmover_skus = set(flags.loc[flags['flag'] == 'SLOW_MOVER', 'sku_id'])

**Rule Based Data** - Check in inventory_snapshot > inventory file:

In [ ]:
our_stockout_skus = set(inventory.loc[inventory['stockout_risk'], 'sku_id'])
our_overstock_skus = set(inventory.loc[inventory['overstock_risk'], 'sku_id'])

**Identify hits** if both are true w.r.t each other

In [ ]:
hit_stockout = known_stockout_skus & our_stockout_skus
hit_slowmover = known_slowmover_skus & our_overstock_skus

**Recalls**

In [ ]:
recall_stockout = len(hit_stockout) / len(known_stockout_skus)
recall_slowmover = len(hit_slowmover) / len(known_slowmover_skus)

**Precison**

In [ ]:
precision_stockout = len(hit_stockout) / len(our_stockout_skus)
precision_slowmover = len(hit_slowmover) / len(our_overstock_skus)

In [ ]:
print(f"STOCKOUT_RISK  recall={recall_stockout:.0%} ({len(hit_stockout)}/{len(known_stockout_skus)} known cases caught)  "
      f"precision={precision_stockout:.1%} ({len(hit_stockout)}/{len(our_stockout_skus)} of our flags were real)")


In [ ]:
print(f"SLOW_MOVER:     recall={recall_slowmover:.0%} ({len(hit_slowmover)}/{len(known_slowmover_skus)} known cases caught)  "
      f"precision={precision_slowmover:.1%} ({len(hit_slowmover)}/{len(our_overstock_skus)} of our flags were real)")


**Insights:** Notice: **recall is high** but **precision is very low** — the rule catches every real case but also flags thousands of SKUs that aren't actually at risk. A single day stock snapshot compared to `reorder_point` isn't enough on its own.

In Week 2's forecast-driven risk score (using actual sales velocity, not just a static threshold) will help a lot to make it better.